# 📕 Notebook 4 — Training Pipeline
## Focal Loss · Trainer · Early Stopping

This notebook implements the full training loop including:
- **Focal Loss** for the proposed BERT-CNN model
- **Weighted Cross-Entropy** for baseline models
- **Differential learning rates** (encoder vs classifier head)
- **Warmup scheduler** with linear decay
- **Early stopping** with best-checkpoint saving
- One-click training for any model

In [ ]:
import os, time, json, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix, classification_report
import pandas as pd
warnings.filterwarnings('ignore')

# ── Full config (paste from NB1 or re-define here) ────────
DATA_PATH      = "suicide_detection.csv"
OUTPUT_DIR     = "outputs"; LOG_DIR = "logs"; CHECKPOINT_DIR = "checkpoints"
TEXT_COLUMN    = "text";  LABEL_COLUMN = "class"
LABEL_MAP      = {"suicide": 1, "non-suicide": 0}
TEST_SIZE      = 0.15;  VAL_SIZE = 0.15;  RANDOM_SEED = 42
MAX_LENGTH     = 512;   BATCH_SIZE = 16;  EPOCHS = 5
LEARNING_RATE  = 2e-5;  WARMUP_RATIO = 0.1;  WEIGHT_DECAY = 0.01
GRAD_CLIP      = 1.0;   EARLY_STOP_PATIENCE = 3
CNN_FILTERS    = 256;   CNN_KERNEL_SIZES = [2,3,4]
DROPOUT_RATE   = 0.3;   FOCAL_LOSS_GAMMA = 2.0;  FOCAL_LOSS_ALPHA = 0.25
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODELS = {
    "BERT":"bert-base-uncased","RoBERTa":"roberta-base",
    "DistilBERT":"distilbert-base-uncased","BERT-CNN":"bert-base-uncased"
}
for d in [OUTPUT_DIR,LOG_DIR,CHECKPOINT_DIR]: os.makedirs(d,exist_ok=True)
print("Config loaded. Device:", DEVICE)

## 4.1 Focal Loss
**Reference:** Lin et al. (2017) — *Focal Loss for Dense Object Detection* (RetinaNet)

$$FL(p_t) = -\alpha_t \cdot (1 - p_t)^\gamma \cdot \log(p_t)$$

- **γ (gamma)**: down-weights easy samples — model focuses on hard misclassifications
- **α (alpha)**: class-balance factor

Used **only for BERT-CNN**. Baselines use weighted CrossEntropy for a fair comparison.

In [ ]:
class FocalLoss(nn.Module):
    """
    Focal Loss — Lin et al. 2017.
    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)

    gamma: focusing parameter (default 2.0)
    alpha: class balance (default 0.25)
    """
    def __init__(self, gamma: float = FOCAL_LOSS_GAMMA,
                 alpha: float = FOCAL_LOSS_ALPHA,
                 reduction: str = "mean"):
        super().__init__()
        self.gamma = gamma; self.alpha = alpha; self.reduction = reduction

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        ce     = F.cross_entropy(logits, targets, reduction="none")
        p_t    = torch.exp(-ce)
        a_t    = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        focal  = a_t.float() * (1 - p_t) ** self.gamma * ce
        return focal.mean() if self.reduction == "mean" else focal.sum()

print("FocalLoss defined.")

## 4.2 Early Stopping

In [ ]:
class EarlyStopping:
    """
    Monitors validation F1. Saves best checkpoint and stops if no
    improvement for `patience` consecutive epochs.
    """
    def __init__(self, patience: int = EARLY_STOP_PATIENCE, min_delta: float = 1e-4):
        self.patience = patience; self.min_delta = min_delta
        self.best_score = None; self.counter = 0; self.stop = False

    def __call__(self, val_f1: float, model: nn.Module, save_path: str) -> None:
        if self.best_score is None or val_f1 > self.best_score + self.min_delta:
            self.best_score = val_f1; self.counter = 0
            torch.save(model.state_dict(), save_path)
            print(f"    ✔ Best model saved  (val F1 = {val_f1:.4f})")
        else:
            self.counter += 1
            print(f"    EarlyStopping: {self.counter}/{self.patience}")
            if self.counter >= self.patience: self.stop = True

print("EarlyStopping defined.")

## 4.3 Dataset & DataLoader
(Reproduced from Notebook 2 for standalone execution)

In [ ]:
class SuicideDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=MAX_LENGTH):
        self.texts = texts; self.labels = labels
        self.tokenizer = tokenizer; self.max_len = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(str(self.texts[idx]),max_length=self.max_len,
                             padding="max_length",truncation=True,return_tensors="pt")
        return {"input_ids":enc["input_ids"].squeeze(0),
                "attention_mask":enc["attention_mask"].squeeze(0),
                "token_type_ids":enc.get("token_type_ids",
                                  torch.zeros(self.max_len,dtype=torch.long)).squeeze(0),
                "label":torch.tensor(self.labels[idx],dtype=torch.long)}

def load_and_split():
    df = pd.read_csv(DATA_PATH, index_col=0)[[TEXT_COLUMN,LABEL_COLUMN]]
    df.dropna(inplace=True); df.drop_duplicates(subset=[TEXT_COLUMN],inplace=True)
    df[LABEL_COLUMN] = df[LABEL_COLUMN].str.strip().map(LABEL_MAP)
    df.dropna(subset=[LABEL_COLUMN],inplace=True); df[LABEL_COLUMN]=df[LABEL_COLUMN].astype(int)
    X,y = df[TEXT_COLUMN].values, df[LABEL_COLUMN].values
    X_tr,X_tmp,y_tr,y_tmp = train_test_split(X,y,test_size=TEST_SIZE+VAL_SIZE,
                                               stratify=y,random_state=RANDOM_SEED)
    vr = VAL_SIZE/(TEST_SIZE+VAL_SIZE)
    X_v,X_te,y_v,y_te = train_test_split(X_tmp,y_tmp,test_size=1-vr,
                                          stratify=y_tmp,random_state=RANDOM_SEED)
    cw = torch.tensor(compute_class_weight("balanced",classes=np.unique(y_tr),y=y_tr),
                      dtype=torch.float).to(DEVICE)
    print(f"Train:{len(X_tr):,} Val:{len(X_v):,} Test:{len(X_te):,}")
    return (X_tr,y_tr),(X_v,y_v),(X_te,y_te),cw

(X_train,y_train),(X_val,y_val),(X_test,y_test),class_weights = load_and_split()

def get_dataloaders(model_name):
    tok = AutoTokenizer.from_pretrained(model_name)
    return (DataLoader(SuicideDataset(X_train,y_train,tok),batch_size=BATCH_SIZE,shuffle=True,num_workers=2,pin_memory=True),
            DataLoader(SuicideDataset(X_val,y_val,tok),batch_size=BATCH_SIZE,shuffle=False,num_workers=2,pin_memory=True),
            DataLoader(SuicideDataset(X_test,y_test,tok),batch_size=BATCH_SIZE,shuffle=False,num_workers=2,pin_memory=True))

## 4.4 Model Architectures
(Reproduced from Notebook 3 for standalone execution)

In [ ]:
class AttentionGate(nn.Module):
    def __init__(self,d): super().__init__(); self.attn=nn.Linear(d,1)
    def forward(self,x):
        x=x.permute(0,2,1); w=torch.softmax(self.attn(x),dim=1); return (w*x).sum(1)

class MultiScaleCNN(nn.Module):
    def __init__(self,in_ch,out_ch,ks=CNN_KERNEL_SIZES):
        super().__init__()
        self.convs=nn.ModuleList([nn.Conv1d(in_ch,out_ch,k,padding=k//2) for k in ks])
        self.gates=nn.ModuleList([AttentionGate(out_ch) for _ in ks])
        self.bns=nn.ModuleList([nn.BatchNorm1d(out_ch) for _ in ks])
    def forward(self,x):
        x=x.permute(0,2,1)
        return torch.cat([g(F.gelu(bn(c(x)))) for c,g,bn in zip(self.convs,self.gates,self.bns)],dim=-1)

class BaselineClassifier(nn.Module):
    def __init__(self,name,nc=2,drop=DROPOUT_RATE):
        super().__init__(); self.enc=AutoModel.from_pretrained(name)
        self.drop=nn.Dropout(drop); self.clf=nn.Linear(self.enc.config.hidden_size,nc)
    def forward(self,ids,mask,tti=None):
        o=self.enc(input_ids=ids,attention_mask=mask,token_type_ids=tti)
        p=o.pooler_output if hasattr(o,"pooler_output") and o.pooler_output is not None else o.last_hidden_state[:,0]
        return self.clf(self.drop(p))

class BertCNNClassifier(nn.Module):
    def __init__(self,name="bert-base-uncased",nc=2,f=CNN_FILTERS,ks=CNN_KERNEL_SIZES,drop=DROPOUT_RATE):
        super().__init__(); self.enc=AutoModel.from_pretrained(name)
        h=self.enc.config.hidden_size; cd=f*len(ks); fd=h+cd
        self.cnn=MultiScaleCNN(h,f,ks); self.ln=nn.LayerNorm(fd); self.drop=nn.Dropout(drop)
        self.clf=nn.Sequential(nn.Linear(fd,512),nn.GELU(),nn.Dropout(drop),
                               nn.Linear(512,128),nn.GELU(),nn.Dropout(drop),nn.Linear(128,nc))
    def forward(self,ids,mask,tti=None):
        o=self.enc(input_ids=ids,attention_mask=mask,token_type_ids=tti)
        cls=o.pooler_output if hasattr(o,"pooler_output") and o.pooler_output is not None else o.last_hidden_state[:,0]
        fused=torch.cat([cls,self.cnn(o.last_hidden_state)],dim=-1)
        return self.clf(self.drop(self.ln(fused)))

def get_model(key):
    m = BertCNNClassifier(MODELS[key]) if key=="BERT-CNN" else BaselineClassifier(MODELS[key])
    return m.to(DEVICE)

print("Model classes ready.")

## 4.5 Training & Evaluation Functions

In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler, loss_fn, epoch):
    model.train()
    total_loss, preds, labels = 0.0, [], []
    t0 = time.time()
    for step, batch in enumerate(loader):
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        tti  = batch["token_type_ids"].to(DEVICE)
        y    = batch["label"].to(DEVICE)
        optimizer.zero_grad()
        logits = model(ids, mask, tti)
        loss   = loss_fn(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step(); scheduler.step()
        total_loss += loss.item()
        preds  += logits.argmax(-1).cpu().tolist()
        labels += y.cpu().tolist()
        if (step+1) % 100 == 0:
            print(f"    step {step+1}/{len(loader)} | loss {loss.item():.4f} | {time.time()-t0:.0f}s")
    return {"loss":total_loss/len(loader),
            "accuracy":accuracy_score(labels,preds),
            "f1":f1_score(labels,preds,average="macro")}

@torch.no_grad()
def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss, preds, labels, probs = 0.0, [], [], []
    for batch in loader:
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        tti  = batch["token_type_ids"].to(DEVICE)
        y    = batch["label"].to(DEVICE)
        logits = model(ids, mask, tti)
        total_loss += loss_fn(logits, y).item()
        probs  += torch.softmax(logits,-1)[:,1].cpu().tolist()
        preds  += logits.argmax(-1).cpu().tolist()
        labels += y.cpu().tolist()
    return {"loss":total_loss/len(loader),
            "accuracy":accuracy_score(labels,preds),
            "precision":precision_score(labels,preds,average="macro"),
            "recall":recall_score(labels,preds,average="macro"),
            "f1":f1_score(labels,preds,average="macro"),
            "f1_suicide":f1_score(labels,preds,pos_label=1,average="binary"),
            "roc_auc":roc_auc_score(labels,probs),
            "cm":confusion_matrix(labels,preds).tolist(),
            "report":classification_report(labels,preds,target_names=["non-suicide","suicide"])}

print("Training & evaluation functions defined.")

## 4.6 Full Trainer
Runs training for any model key. Includes:
- Differential learning rates (encoder 2e-5, head 2e-4)
- Linear warmup scheduler
- Early stopping
- Best checkpoint saving

In [ ]:
def train_model(model_key: str):
    """End-to-end trainer for a single model. Returns test metrics."""
    print(f"\n{'='*55}")
    print(f"  Training: {model_key}")
    print(f"{'='*55}")

    train_loader, val_loader, test_loader = get_dataloaders(MODELS[model_key])
    model    = get_model(model_key)
    loss_fn  = FocalLoss() if model_key=="BERT-CNN" else nn.CrossEntropyLoss(weight=class_weights)
    ckpt     = os.path.join(CHECKPOINT_DIR, f"{model_key}_best.pt")

    # Differential LR: encoder gets LEARNING_RATE, head gets 10x
    no_decay = ["bias","LayerNorm.weight"]
    enc_p  = [(n,p) for n,p in model.named_parameters() if "enc" in n]
    head_p = [(n,p) for n,p in model.named_parameters() if "enc" not in n]
    optimizer = AdamW([
        {"params":[p for n,p in enc_p  if not any(nd in n for nd in no_decay)],"lr":LEARNING_RATE,"weight_decay":WEIGHT_DECAY},
        {"params":[p for n,p in enc_p  if     any(nd in n for nd in no_decay)],"lr":LEARNING_RATE,"weight_decay":0.0},
        {"params":[p for n,p in head_p if not any(nd in n for nd in no_decay)],"lr":LEARNING_RATE*10,"weight_decay":WEIGHT_DECAY},
        {"params":[p for n,p in head_p if     any(nd in n for nd in no_decay)],"lr":LEARNING_RATE*10,"weight_decay":0.0},
    ])
    total_steps  = len(train_loader) * EPOCHS
    scheduler    = get_linear_schedule_with_warmup(optimizer,int(total_steps*WARMUP_RATIO),total_steps)
    stopper      = EarlyStopping()
    history      = {"train":[],"val":[]}

    for epoch in range(1, EPOCHS+1):
        print(f"\n  Epoch {epoch}/{EPOCHS}")
        tr = train_one_epoch(model, train_loader, optimizer, scheduler, loss_fn, epoch)
        vl = evaluate(model, val_loader, loss_fn)
        print(f"  Train → loss:{tr['loss']:.4f}  f1:{tr['f1']:.4f}")
        print(f"  Val   → loss:{vl['loss']:.4f}  f1:{vl['f1']:.4f}  auc:{vl['roc_auc']:.4f}")
        history["train"].append(tr); history["val"].append(vl)
        stopper(vl["f1"], model, ckpt)
        if stopper.stop: print("  Early stopping."); break

    model.load_state_dict(torch.load(ckpt))
    test_m = evaluate(model, test_loader, loss_fn)

    print(f"\n  ── Test Results ──")
    for k in ["accuracy","precision","recall","f1","f1_suicide","roc_auc"]:
        print(f"    {k:<15}: {test_m[k]:.4f}")
    print(f"\n{test_m['report']}")

    # Save history and results
    with open(os.path.join(LOG_DIR,f"{model_key}_history.json"),"w") as f:
        json.dump(history, f, indent=2)
    res = {k:v for k,v in test_m.items() if k!="report"}
    with open(os.path.join(OUTPUT_DIR,f"results_{model_key}.json"),"w") as f:
        json.dump(res, f, indent=2)

    del model; torch.cuda.empty_cache()
    return test_m, history

print("Trainer ready.")

## 4.7 Run Training
Choose which models to train. For a full research run, train all 4.
> ⏱ **Expected time per model** on a single GPU (T4/A100): ~30–90 min depending on dataset size.

In [ ]:
# ── Choose which models to train ──────────────────────────
# Options: "BERT", "RoBERTa", "DistilBERT", "BERT-CNN"
TRAIN_MODELS = ["BERT", "RoBERTa", "DistilBERT", "BERT-CNN"]

all_results  = {}
all_histories = {}

for model_key in TRAIN_MODELS:
    metrics, history = train_model(model_key)
    all_results[model_key]   = metrics
    all_histories[model_key] = history

print("\n✓ All models trained.")